# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id`.

In [ ]:
# Inspect all record sets and their fields
record_set_ids = [rs['@id'] for rs in metadata.as_jsonld().get('recordSet', [])]
print('Record Sets in the dataset:')
for rs in metadata.as_jsonld().get('recordSet', []):
    print(f"- Record Set @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, list their fields with @id
for rs in metadata.as_jsonld().get('recordSet', []):
    print(f"\nFields in Record Set {rs['@id']}:")
    for field in rs.get('field', []):
        fname = field.get('name', 'N/A')
        print(f"  - Field @id: {field['@id']}, name: {fname}")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as shown above.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in metadata.as_jsonld().get('recordSet', [])]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from Record Set: {record_set_id}")

# If there is at least one record set, preview the first one
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    print(f"\nPreview of records from {first_rs}:")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll demonstrate on the first record set loaded.

In [ ]:
# Select numeric and group fields by @id. Adjust these as appropriate for your data overview.
first_record_set = record_set_ids[0] if record_set_ids else None
df = dataframes.get(first_record_set)

if df is not None and not df.empty:
    # Select a numeric field (guess column names, e.g. 'Molecular Weight' or 'PeptideCount')
    possible_numeric_fields = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[c])]
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
    print(f"Selected numeric field for demonstration: {numeric_field}")
    
    threshold = df[numeric_field].mean() # Example: filter above mean
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a field if available
    # Guess group field
    group_fields = [c for c in df.columns if c.lower() in ['sample', 'group', 'accession', 'description']]
    group_field = group_fields[0] if group_fields else None
    
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}':")
        display(grouped_df.head())
else:
    print("No records available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot histograms and scatter plots for numeric fields if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and possible_numeric_fields:
    # Histogram for the numeric field
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If there is more than one numeric field, plot a scatter plot
    if len(possible_numeric_fields) > 1:
        plt.figure()
        sns.scatterplot(data=df, x=possible_numeric_fields[0], y=possible_numeric_fields[1])
        plt.title(f"Scatter plot: {possible_numeric_fields[0]} vs {possible_numeric_fields[1]}")
        plt.xlabel(possible_numeric_fields[0])
        plt.ylabel(possible_numeric_fields[1])
        plt.show()
else:
    print("Not enough numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR^2 mass spectrometry dataset via its Croissant schema using `mlcroissant`
- Reviewed its data structure by record set and fields (referenced by their `@id`)
- Loaded records into DataFrames for analysis
- Applied basic filtering, normalization, and grouped analysis
- Visualized numeric distributions and relationships

For further analysis, consider investigating relationships across multiple record sets or integrating additional metadata fields as referenced by their `@id` fields.